In [1]:
import pandas as pd
import numpy as np
from collections import defaultdict
import json
from rdkit.Chem import AllChem as Chem
from rdkit.Chem import rdMolDescriptors
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')


In [2]:
pubchem = pd.read_csv("../../data/pubchemlite/PubChemLite_CCSbase_20260529.csv")
column_names = pubchem.columns
print(column_names)

Index(['Identifier', 'FirstBlock', 'PubMed_Count', 'Patent_Count',
       'Related_CIDs', 'Synonym', 'MolecularFormula', 'SMILES', 'InChI',
       'InChIKey', 'MonoisotopicMass', 'XLogP', 'CompoundName',
       'AnnoTypeCount', 'AgroChemInfo', 'BioPathway', 'DrugMedicInfo',
       'FoodRelated', 'PharmacoInfo', 'SafetyInfo', 'ToxicityInfo', 'KnownUse',
       'DisorderDisease', 'Identification', 'ChemClass', 'pred_CCS_A2_[M+H]+',
       'pred_CCS_A2_[M+Na]+', 'pred_CCS_A2_[M-H]-', 'pred_CCS_A2_[M+NH4]+',
       'pred_CCS_A2_[M+K]+', 'pred_CCS_A2_[M+H-H2O]+', 'pred_CCS_A2_[M+HCOO]-',
       'pred_CCS_A2_[M+CH3COO]-', 'pred_CCS_A2_[M+Na-2H]-', 'pred_CCS_A2_[M]+',
       'pred_CCS_A2_[M]-'],
      dtype='object')


In [3]:
pubchem.iloc[0:5, 0:30]

,Identifier,FirstBlock,PubMed_Count,Patent_Count,Related_CIDs,Synonym,MolecularFormula,SMILES,InChI,InChIKey,...,ToxicityInfo,KnownUse,DisorderDisease,Identification,ChemClass,pred_CCS_A2_[M+H]+,pred_CCS_A2_[M+Na]+,pred_CCS_A2_[M-H]-,pred_CCS_A2_[M+NH4]+,pred_CCS_A2_[M+K]+
0,3,INCSWYKICIYAHB,11,199,9964159 23615184 45266758,"(5S,6S)-5,6-dihydroxycyclohexa-1,3-diene-1-car...",C7H8O4,C1=CC(C(C(=C1)C(=O)O)O)O,InChI=1S/C7H8O4/c8-5-3-1-2-4(6(5)9)7(10)11/h1-...,INCSWYKICIYAHB-UHFFFAOYSA-N,...,0,0,0,0,0,128.6,136.2,128.7,147.6,134.3
1,4,HXKKHQJGJAFBHI,77,114242,4 111033 439938 446260 4631415 7311736,1-Aminopropan-2-ol,C3H9NO,CC(CN)O,"InChI=1S/C3H9NO/c1-3(5)2-4/h3,5H,2,4H2,1H3",HXKKHQJGJAFBHI-UHFFFAOYSA-N,...,3,4,0,2,5,114.0,120.9,112.8,137.0,121.1
2,5,HIQNVODXENYOFK,3,11,5 25244426,3-Amino-2-oxopropyl phosphate,C3H8NO5P,C(C(=O)COP(=O)(O)O)N,"InChI=1S/C3H8NO5P/c4-1-3(5)2-9-10(6,7)8/h1-2,4...",HIQNVODXENYOFK-UHFFFAOYSA-N,...,0,0,0,0,0,133.1,139.7,129.3,151.8,139.9
3,6,VYZAHLCBVHPDDF,3381,15879,6,"1-chloro-2,4-dinitrobenzene",C6H3ClN2O4,C1=CC(=C(C=C1[N+](=O)[O-])[N+](=O)[O-])Cl,InChI=1S/C6H3ClN2O4/c7-5-2-1-4(8(10)11)3-6(5)9...,VYZAHLCBVHPDDF-UHFFFAOYSA-N,...,3,6,1,2,3,138.9,146.7,142.6,156.5,136.5
4,8,PDGXJDXVGMHUIR,4,577,8 448154 21145376 21263701,"2,3-Dihydroxy-3-methylpentanoic acid",C6H12O4,CCC(C)(C(C(=O)O)O)O,"InChI=1S/C6H12O4/c1-3-6(2,10)4(7)5(8)9/h4,7,10...",PDGXJDXVGMHUIR-UHFFFAOYSA-N,...,0,0,0,0,3,131.0,137.1,127.4,150.2,136.8


In [4]:
# Filtering the dataset to only keep molecules lighter than 1000 Da
# and molecules that have at least one bond (i.e. not single atoms)
def is_single_atom(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return True  # Treat invalid SMILES as single atoms
    return mol.GetNumBonds() == 0
#drop rows where MonisotopicMass above 1000
pubchem = pubchem[pubchem["MonoisotopicMass"] <= 1000]
pubchem = pubchem[~pubchem["SMILES"].apply(is_single_atom)]
print(f"Number of compounds after filtering: {len(pubchem)}")

Number of compounds after filtering: 530797


## Create default Precursor masses
- computed based on the reported pubchemlite offset

In [5]:
def compute_precursor_mzs(mass):
    """Return dict of adduct -> theoretical m/z for a given neutral monoisotopic mass."""
    offsets = {
        "[M+H]+":       1.007276,
        "[M+Na]+":     22.989218,
        "[M-H]-":      -1.007276,
        "[M+NH4]+":    18.033823,
        "[M+K]+":      38.963158,
        "[M+H-H2O]+": -17.002740,
        "[M+HCOO]-":   44.998201,
        "[M+CH3COO]-": 59.013851,
        "[M+Na-2H]-":  20.974666,
        "[M]+":        -0.00054858,
        "[M]-":         0.00054858,
    }
    
    return {mass + offset for offset in offsets.values()}

In [6]:
#create a table with precursor m/z in first column and the corresponding SMILES in second column
# Create the expanded table
# Build the expanded table
rows = []
mass_smiles = pubchem[["MonoisotopicMass", "SMILES"]]

for mass, smiles in mass_smiles.itertuples(index=False):
    for mz in compute_precursor_mzs(mass):
        rows.append({"Precursor_mz": mz, "SMILES": smiles})

pubchem_precursor_mz = pd.DataFrame(rows)

print(f"Number of precursor m/z entries: {len(pubchem_precursor_mz)}")
print(f"Expected: 11 * {len(mass_smiles)} = {11 * len(mass_smiles)}")
print(f"Match: {len(pubchem_precursor_mz) == 11 * len(mass_smiles)}")

Number of precursor m/z entries: 5838767
Expected: 11 * 530797 = 5838767
Match: True


In [12]:
print(f"unique masses in pubchem_precursor_mz: {pubchem_precursor_mz['Precursor_mz'].nunique()}")

unique masses in pubchem_precursor_mz: 1551433


## MassSpecGym Algorithmus:
- first step select lightest candidate L which is not yet binned
- second step find centroid C select next heavier molecule 
    compute ppm range of the molecule if range includes L check next heavier molecule, until L is out of range at C+1, so select C.
- third step select all molecules heavier than C that still fall in the  ppm range
- create a 1ppm bin for this centroid
- start first step


In [7]:
def mass_spec_binning(data, ppm=3):
    """
    data contains a DataFrame with the first column as the mass and the second column as the SMILES"""
    

    # --- 1. Load and sort masses + smiles ---
    mass_smiles = lambda df: df.sort_values(df.columns[0])

    masses = mass_smiles(data).iloc[:, 0].values
    smiles = mass_smiles(data).iloc[:, 1].values

    N = len(masses)

    # Track which molecules are already assigned to a bin
    assigned = np.zeros(N, dtype=bool)

    bins = dict() # dictionary where key is the centroid mass and value is a list of smiles
    #bins = []   # list of bins, each bin is list of (mass, smiles)

    ppm = 3  # 1 ppm window


    def ppm_range(center_mass, ppm):
        """Return absolute Da window for ±ppm around center_mass."""
        delta = center_mass * ppm * 1e-6
        return center_mass - delta, center_mass + delta


    i = 0
    while i < N:
        # Step 1: find the lightest unassigned molecule L
        if assigned[i]:
            i += 1
            continue

        L_mass = masses[i]

        # Step 2: find centroid C
        # Start scanning heavier molecules until L is out of range
        C_index = i
        j = i + 1

        while j < N:
            C_mass = masses[j]
            low, high = ppm_range(C_mass, ppm)

            if low <= L_mass <= high:
                # L still inside ppm window → continue scanning
                C_index = j
                j += 1
            else:
                # L is out of range → previous j-1 is the centroid
                break

        # Final centroid mass
        C_mass = masses[C_index]
        low, high = ppm_range(C_mass, ppm)

        # Step 3: collect all molecules heavier than C that still fall in the ppm window
        bin_members = []

        # include centroid and all lighter ones that fall inside window
        k = i
        while k < N and masses[k] <= high:
            if not assigned[k] and low <= masses[k] <= high:
                #bin_members.append((masses[k], smiles[k]))
                bin_members.append(smiles[k])
                assigned[k] = True
            k += 1

        #bins.append(bin_members)
        bins[C_mass] = bin_members
        # Continue from next unassigned molecule
        i += 1
    
    # bins is now a list of dynamic-ppm bins
    return bins


In [8]:
#bins = mass_spec_binning(mass_smiles)
bins = mass_spec_binning(pubchem_precursor_mz)

print(f"Total bins: {len(bins)}")
print(f"Average bin size: {np.mean([len(b) for b in bins.values()]):.2f}")
print(f"Medidan bin size: {np.median([len(b) for b in bins.values()])}")
print(f"Amount of singletons: {sum(len(b) == 1 for b in bins.values())}")
print(f"Maximum bin size: {max(len(b) for b in bins.values())}")
# half of the bins are singletons, which is expected
# with a small dataset like pubchem light

Total bins: 194601
Average bin size: 30.00
Medidan bin size: 9.0
Amount of singletons: 33418
Maximum bin size: 998


In [9]:
json.dump(bins, open("../../data/pubchemlite/precursor_bins.json", "w"), indent=2)

In [10]:
# select only the bins after 992.5561364700001
largebins = dict()
for bin in bins:
    if bin > 992.5561364700001:
        largebins[bin] = bins[bin]

In [11]:
json.dump(largebins, open("../../data/pubchemlite/large_precursor_bins.json", "w"), indent=2)